<a href="https://colab.research.google.com/github/Ashwini9713/threat-intelligence/blob/main/exp15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
from urllib.parse import urlparse

SUSPICIOUS_KEYWORDS = ["login", "verify", "update", "secure", "account", "banking", "confirm"]
SHORTENERS = ["bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly"]

def has_ip_address(url):
    pattern = r"(https?://)?(\d{1,3}\.){3}\d{1,3}"
    return bool(re.match(pattern, url))

def count_subdomains(hostname):
    return hostname.count(".")

def analyze_url(url):
    score = 0
    reasons = []
    parsed = urlparse(url if url.startswith("http") else "http://" + url)
    hostname = parsed.netloc or parsed.path

    if has_ip_address(url):
        score += 3
        reasons.append("Uses raw IP address instead of domain")

    if "@" in url:
        score += 2
        reasons.append("Contains '@' symbol (redirection trick)")

    if url.count("-") > 3:
        score += 1
        reasons.append("Excessive hyphens in domain")

    if count_subdomains(hostname) > 3:
        score += 2
        reasons.append("Too many subdomains")

    if any(short in hostname for short in SHORTENERS):
        score += 2
        reasons.append("Uses a known URL shortener")

    if any(word in url.lower() for word in SUSPICIOUS_KEYWORDS):
        score += 1
        reasons.append("Contains suspicious keyword")

    if len(url) > 75:
        score += 1
        reasons.append("Unusually long URL")

    if not url.startswith("https"):
        score += 1
        reasons.append("No HTTPS encryption")

    verdict = "PHISHING SUSPECTED" if score >= 4 else "LIKELY SAFE"
    return verdict, score, reasons

if __name__ == "__main__":
    url = input("Enter a URL to check: ")
    verdict, score, reasons = analyze_url(url)
    print(f"\nVerdict: {verdict} (risk score: {score})")
    if reasons:
        print("Reasons:")
        for r in reasons:
            print(f" - {r}")

Enter a URL to check: http://secure-login-update.bit.ly/verify-account@paypal.com 

Verdict: PHISHING SUSPECTED (risk score: 6)
Reasons:
 - Contains '@' symbol (redirection trick)
 - Uses a known URL shortener
 - Contains suspicious keyword
 - No HTTPS encryption
